# t-SNE: subsample convergence to the full token distribution

Load BigBird layer-0 embeddings on a long Wikipedia sample, compute a single t-SNE on the full set (`TSNE_BASE_SIZE` points), and overlay independent random subsamples of various sizes to visualise how subsamples progressively cover the support of the token distribution.

Set `SMOKE_TEST = True` for a fast local check (small base size, tiny perplexity).


In [ ]:
from pathlib import Path
import pickle
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE

# Walk up from the current working directory until we find pyproject.toml —
# robust whether the notebook is launched from the repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tsc import get_layer0_embeddings, load_model, load_text, load_tokenizer


In [ ]:
SMOKE_TEST = False    # set True for a tiny local run

if SMOKE_TEST:
    TSNE_BASE_SIZE = 512
    SUBSAMPLE_SIZES = [32, 128]
    TSNE_PERPLEXITY = 15
    TSNE_MAX_ITER = 250
else:
    TSNE_BASE_SIZE = 50_000
    SUBSAMPLE_SIZES = [3_000, 10_000]
    TSNE_PERPLEXITY = 50
    TSNE_MAX_ITER = 1000

MODEL_KEY = 'bigbird-base'
RANDOM_SEED = 42

OUTPUT_DIR = REPO_ROOT / 'figures'
CACHE_DIR = REPO_ROOT / 'data' / 'tsne_cache'
CACHE_FILE = CACHE_DIR / f'tsne_base_{TSNE_BASE_SIZE}.pkl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_embeddings(n_tokens):
    """BigBird layer-0 embeddings for n_tokens taken from WikiText (English)."""
    text = load_text(max_length=n_tokens, source='wiki', language='en')
    model = load_model(MODEL_KEY, attention_type='block_sparse')
    tokenizer = load_tokenizer(MODEL_KEY)
    tokens = tokenizer(text, return_tensors='pt', truncation=True, max_length=n_tokens)
    tokens = {k: v.to(model.device) for k, v in tokens.items()}
    with torch.no_grad():
        X = get_layer0_embeddings(model, tokens)
    return X.squeeze(0).cpu().numpy()


In [ ]:
if CACHE_FILE.exists():
    with open(CACHE_FILE, 'rb') as f:
        cached = pickle.load(f)
    embeddings_2d = cached['embeddings_2d']
    n_points = cached['n_points']
    print(f'loaded cached t-SNE: {embeddings_2d.shape}')
else:
    embeddings = load_embeddings(TSNE_BASE_SIZE)
    n_points = embeddings.shape[0]
    print(f'running t-SNE on {n_points} points (perplexity={TSNE_PERPLEXITY}, max_iter={TSNE_MAX_ITER})')
    embeddings_2d = TSNE(
        n_components=2,
        perplexity=TSNE_PERPLEXITY,
        max_iter=TSNE_MAX_ITER,
        random_state=RANDOM_SEED,
        init='pca',
        learning_rate='auto',
        n_jobs=-1,
    ).fit_transform(embeddings)
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump({'embeddings_2d': embeddings_2d, 'n_points': n_points}, f)
    print(f'cached t-SNE to {CACHE_FILE}')


In [ ]:
COLORS = {
    'small':  '#C7E9F1',   # light cyan
    'medium': '#0077B6',   # ocean blue
    'full':   '#001845',   # very dark blue
}
rng = np.random.default_rng(RANDOM_SEED + 1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
           c=COLORS['full'], s=18, alpha=0.85, rasterized=True)
for n, key in zip(sorted(SUBSAMPLE_SIZES, reverse=True), ('medium', 'small')):
    if n > n_points:
        continue
    idx = rng.choice(n_points, size=n, replace=False)
    ax.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
               c=COLORS[key], s=8 if key == 'small' else 14,
               alpha=0.55, rasterized=True)

ax.set_aspect('equal', adjustable='datalim')
ax.axis('off')

handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=COLORS['small'],  markersize=12,
           label=f'n = {sorted(SUBSAMPLE_SIZES)[0]:,}'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=COLORS['medium'], markersize=12,
           label=f'n = {sorted(SUBSAMPLE_SIZES)[-1]:,}'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=COLORS['full'],   markersize=12,
           label=f'n = {n_points:,} (full)'),
]
ax.legend(handles=handles, loc='upper right', frameon=True, framealpha=0.9,
          fontsize=13, markerscale=1.1)

fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(OUTPUT_DIR / f'tsne_subsample_convergence.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print(f'saved {OUTPUT_DIR}/tsne_subsample_convergence.{{png,pdf}}')
